# Library Imports

In [187]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_max_pool
from torch_geometric.nn import SAGEConv

INPUT_FILE      = "dataset.csv"
CHECKPOINT_DIR  = "checkpoints"
FINAL_MODEL     = "model_final.pth"

INPUTS   = ["gain_max_dB", "pm", "gbw"]
OUTPUTS  = ["w1", "w3"]

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.20
TEST_RATIO = 0.10

EPOCHS        = 100
BATCH_SIZE    = 64
LR            = 1e-3
CHECKPOINT_EVERY = 5
ACC_THRESHOLD = 0.25

SEED = 42

# Loading Dataset

In [154]:
def create_graph(x_row, y_row):
    x = torch.tensor(x_row, dtype=torch.float32).view(-1, 1) # node features
    edge_index = torch.tensor([
        [0, 1, 1, 2],
        [1, 0, 2, 1]
    ], dtype=torch.long) # fully connected edges

    y = torch.tensor(y_row, dtype=torch.float32).view(1, -1) # target values

    return Data(x=x, edge_index=edge_index, y=y)

In [155]:
df = pd.read_csv(INPUT_FILE).sample(frac=1, random_state=SEED).reset_index(drop=True)

X = df[INPUTS].values.astype(np.float32)
y = df[OUTPUTS].values.astype(np.float32)

graphs = [create_graph(X[i], y[i]) for i in range(len(X))]

n = len(graphs)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train_set = graphs[:n_train]
val_set = graphs[n_train:n_train+n_val]
test_set = graphs[n_train+n_val:]

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

/tmp/ipykernel_107292/129201472.py:16: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
/tmp/ipykernel_107292/129201472.py:17: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
/tmp/ipykernel_107292/129201472.py:18: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)


# Model Class Definition

In [156]:
class SpecNetGNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = GCNConv(1, 128)
        self.conv2 = GCNConv(128, 256)
        self.conv3 = GCNConv(256, 128)

        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 2)

        self.relu = nn.LeakyReLU()
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.relu(self.conv1(x, edge_index))
        x = self.relu(self.conv2(x, edge_index))
        x = self.relu(self.conv3(x, edge_index))

        # graph-level pooling
        x = global_max_pool(x, batch)

        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Training/Evalution

In [157]:
def evaluate(loader):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_elements = 0

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            preds = model(data)

            yb = data.y

            total_loss += criterion(preds, yb).item() * len(yb)

            target_range = yb.max() - yb.min()
            threshold = ACC_THRESHOLD * target_range if target_range > 0 else ACC_THRESHOLD

            correct = (torch.abs(preds - yb) < threshold).float().sum()
            total_correct += correct.item()
            total_elements += yb.numel()
    
    avg_loss = total_loss / len(loader.dataset)
    acc = 100 * total_correct / total_elements

    return avg_loss, acc

In [158]:
model = SpecNetGNN().to(device)
optim = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode='min', factor=0.5, patience=5)

train_acc_history = []
val_acc_history = []

for epoch in range(EPOCHS):
    model.train()

    for data in train_loader:
        data = data.to(device)

        optim.zero_grad()
        preds = model(data)
        loss = criterion(preds, data.y)
        loss.backward()
        optim.step()

    train_mse, train_acc = evaluate(train_loader)
    val_mse, val_acc = evaluate(val_loader)
    scheduler.step(val_mse)

    train_acc_history.append(train_acc)
    val_acc_history.append(val_acc)

    print(f"Epoch {epoch}: Train Acc={train_acc:.2f} | Val Acc={val_acc:.2f}")

/home/arnav-patil/.local/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 0: Train Acc=0.00 | Val Acc=0.00
Epoch 1: Train Acc=0.00 | Val Acc=0.00
Epoch 2: Train Acc=0.00 | Val Acc=0.00
Epoch 3: Train Acc=0.00 | Val Acc=0.00
Epoch 4: Train Acc=0.00 | Val Acc=0.00
Epoch 5: Train Acc=0.75 | Val Acc=0.68
Epoch 6: Train Acc=0.73 | Val Acc=0.51
Epoch 7: Train Acc=0.50 | Val Acc=1.02
Epoch 8: Train Acc=0.63 | Val Acc=0.80
Epoch 9: Train Acc=0.71 | Val Acc=0.62
Epoch 10: Train Acc=0.71 | Val Acc=0.80
Epoch 11: Train Acc=0.68 | Val Acc=0.62
Epoch 12: Train Acc=0.45 | Val Acc=0.51
Epoch 13: Train Acc=0.55 | Val Acc=0.68
Epoch 14: Train Acc=0.58 | Val Acc=0.97
Epoch 15: Train Acc=0.67 | Val Acc=1.14
Epoch 16: Train Acc=0.86 | Val Acc=0.85
Epoch 17: Train Acc=0.57 | Val Acc=0.62
Epoch 18: Train Acc=0.58 | Val Acc=0.45
Epoch 19: Train Acc=0.80 | Val Acc=0.91
Epoch 20: Train Acc=0.50 | Val Acc=1.08
Epoch 21: Train Acc=1.01 | Val Acc=1.02
Epoch 22: Train Acc=0.68 | Val Acc=0.68
Epoch 23: Train Acc=0.83 | Val Acc=1.25
Epoch 24: Train Acc=0.73 | Val Acc=0.80
Epoch 25: 

In [192]:
test_mse, test_acc = evaluate(test_loader)
print(f"Test Acc: {test_acc:.4f} | Test MSE: {test_mse:.6f}")

Test Acc: 0.5682 | Test MSE: 0.009577
